# Basics para poder armarlo (Se reemplaza por modulo 1)

In [1]:
# Importar dataset
import polars as pl
df = pl.read_csv("sell-in.txt.gz", separator="\t")
print(df)

shape: (2_945_818, 7)
┌─────────┬─────────────┬────────────┬─────────────────┬────────────────┬────────────────┬─────────┐
│ periodo ┆ customer_id ┆ product_id ┆ plan_precios_cu ┆ cust_request_q ┆ cust_request_t ┆ tn      │
│ ---     ┆ ---         ┆ ---        ┆ idados          ┆ ty             ┆ n              ┆ ---     │
│ i64     ┆ i64         ┆ i64        ┆ ---             ┆ ---            ┆ ---            ┆ f64     │
│         ┆             ┆            ┆ i64             ┆ i64            ┆ f64            ┆         │
╞═════════╪═════════════╪════════════╪═════════════════╪════════════════╪════════════════╪═════════╡
│ 201701  ┆ 10234       ┆ 20524      ┆ 0               ┆ 2              ┆ 0.053          ┆ 0.053   │
│ 201701  ┆ 10032       ┆ 20524      ┆ 0               ┆ 1              ┆ 0.13628        ┆ 0.13628 │
│ 201701  ┆ 10217       ┆ 20524      ┆ 0               ┆ 1              ┆ 0.03028        ┆ 0.03028 │
│ 201701  ┆ 10125       ┆ 20524      ┆ 0               ┆ 1           

In [2]:
# Por cada producto, se evalua cual es el período más antiguo en el que se vendió este producto para algún cliente. Luego se agrega este dato en una columna llamado "first_sell_in_period" manteniendo el dataset original. Para esto se puede usar la función groupby y agg de polars.
df = df.with_columns(
    pl.col("periodo").min().over("product_id").alias("first_sell_in_period")
)  
print(df)

shape: (2_945_818, 8)
┌─────────┬─────────────┬────────────┬────────────┬────────────┬────────────┬─────────┬────────────┐
│ periodo ┆ customer_id ┆ product_id ┆ plan_preci ┆ cust_reque ┆ cust_reque ┆ tn      ┆ first_sell │
│ ---     ┆ ---         ┆ ---        ┆ os_cuidado ┆ st_qty     ┆ st_tn      ┆ ---     ┆ _in_period │
│ i64     ┆ i64         ┆ i64        ┆ s          ┆ ---        ┆ ---        ┆ f64     ┆ ---        │
│         ┆             ┆            ┆ ---        ┆ i64        ┆ f64        ┆         ┆ i64        │
│         ┆             ┆            ┆ i64        ┆            ┆            ┆         ┆            │
╞═════════╪═════════════╪════════════╪════════════╪════════════╪════════════╪═════════╪════════════╡
│ 201701  ┆ 10234       ┆ 20524      ┆ 0          ┆ 2          ┆ 0.053      ┆ 0.053   ┆ 201701     │
│ 201701  ┆ 10032       ┆ 20524      ┆ 0          ┆ 1          ┆ 0.13628    ┆ 0.13628 ┆ 201701     │
│ 201701  ┆ 10217       ┆ 20524      ┆ 0          ┆ 1          ┆ 0.03

In [3]:
# Para cada combinación de producto y cliente se completan los períodos faltantes entre el período más antiguo (first_sell_in_period) y el período más reciente (201912).
# Hay que completar la columna periodo con el periodo faltante correspondiente en el formato AAAAMM.
# Las columnas customer_id, product_id se mantienen constantes para cada combinación de producto y cliente.  
# Las columnas plan_precios_cuidados y first_sell_in_period se completan con el valor correspondiente a cada producto
# Las columnas cust_request_qty, cust_request_tn, tn se completan con el valor 0.

# Definimos el límite superior de los periodos como entero
FECHA_LIMITE = 201912

# 1. Creamos la grilla con todos los periodos teóricos en formato AAAAMM
grilla_base = (
    df.select(["customer_id", "product_id", "first_sell_in_period"])
    .unique()
    .filter(pl.col("first_sell_in_period") <= FECHA_LIMITE)
    .with_columns(
        # Convertimos AAAAMM a una escala lineal de meses: (Año * 12) + Mes
        inicio_meses=(pl.col("first_sell_in_period") // 100) * 12 + (pl.col("first_sell_in_period") % 100),
        fin_meses=(FECHA_LIMITE // 100) * 12 + (FECHA_LIMITE % 100)
    )
    .with_columns(
        # Generamos el rango de pasos de meses a rellenar
        pasos=pl.int_ranges(0, pl.col("fin_meses") - pl.col("inicio_meses") + 1)
    )
    .explode("pasos")
)

# Reconstruimos la columna 'periodo' en formato entero AAAAMM
grilla = grilla_base.with_columns(
    mes_actual_lineal=pl.col("inicio_meses") + pl.col("pasos")
).with_columns(
    periodo=(
        ((pl.col("mes_actual_lineal") - 1) // 12) * 100 + 
        ((pl.col("mes_actual_lineal") - 1) % 12 + 1)
    ).cast(pl.Int64)
).select(["customer_id", "product_id", "periodo"])

# 2. Hacemos el Left Join con tus datos originales
df_completado = grilla.join(df, on=["customer_id", "product_id", "periodo"], how="left")

# 3. Rellenamos las columnas según tus especificaciones
df_final = df_completado.with_columns(
    # Columnas que van obligatoriamente en 0 si no existía registro
    pl.col("cust_request_qty").fill_null(0),
    pl.col("cust_request_tn").fill_null(0.0),
    pl.col("tn").fill_null(0.0),
    
    # Columnas que mantienen su valor característico por combinación producto-cliente
    pl.col("plan_precios_cuidados").forward_fill().backward_fill().over(["customer_id", "product_id"]),
    pl.col("first_sell_in_period").forward_fill().backward_fill().over(["customer_id", "product_id"])
)

# 4. Si tienes más columnas descriptivas (ej: 'match'), las arrastramos dinámicamente
columnas_restantes = [col for col in df.columns if col not in ["customer_id", "product_id", "periodo", "cust_request_qty", "cust_request_tn", "tn", "plan_precios_cuidados", "first_sell_in_period"]]

if columnas_restantes:
    df_final = df_final.with_columns(
        pl.col(columnas_restantes).forward_fill().backward_fill().over(["customer_id", "product_id"])
    )

# Ordenamos el dataframe para su presentación final
df_final = df_final.sort(["product_id", "customer_id", "periodo"])

print(df_final)

shape: (11_644_324, 8)
┌────────────┬────────────┬─────────┬────────────┬────────────┬────────────┬───────────┬───────────┐
│ customer_i ┆ product_id ┆ periodo ┆ plan_preci ┆ cust_reque ┆ cust_reque ┆ tn        ┆ first_sel │
│ d          ┆ ---        ┆ ---     ┆ os_cuidado ┆ st_qty     ┆ st_tn      ┆ ---       ┆ l_in_peri │
│ ---        ┆ i64        ┆ i64     ┆ s          ┆ ---        ┆ ---        ┆ f64       ┆ od        │
│ i64        ┆            ┆         ┆ ---        ┆ i64        ┆ f64        ┆           ┆ ---       │
│            ┆            ┆         ┆ i64        ┆            ┆            ┆           ┆ i64       │
╞════════════╪════════════╪═════════╪════════════╪════════════╪════════════╪═══════════╪═══════════╡
│ 10001      ┆ 20001      ┆ 201701  ┆ 0          ┆ 11         ┆ 99.43861   ┆ 99.43861  ┆ 201701    │
│ 10001      ┆ 20001      ┆ 201702  ┆ 0          ┆ 23         ┆ 198.84365  ┆ 198.84365 ┆ 201701    │
│ 10001      ┆ 20001      ┆ 201703  ┆ 0          ┆ 33         ┆ 92.4

In [4]:
# Se carga la base de datos tb_productos.txt como un dataframe de polars y se muestra su contenido (el separador es tabulación)
df_productos = pl.read_csv("tb_productos.txt", separator="\t")
print(df_productos)

shape: (1_251, 7)
┌───────┬──────────┬─────────────┬──────────┬──────────┬────────────┬───────────────────┐
│ cat1  ┆ cat2     ┆ cat3        ┆ brand    ┆ sku_size ┆ product_id ┆ descripcion       │
│ ---   ┆ ---      ┆ ---         ┆ ---      ┆ ---      ┆ ---        ┆ ---               │
│ str   ┆ str      ┆ str         ┆ str      ┆ i64      ┆ i64        ┆ str               │
╞═══════╪══════════╪═════════════╪══════════╪══════════╪════════════╪═══════════════════╡
│ FOODS ┆ ADEREZOS ┆ Aji Picante ┆ NATURA   ┆ 240      ┆ 20609      ┆ Salsa Aji Picante │
│ FOODS ┆ ADEREZOS ┆ Barbacoa    ┆ NATURA   ┆ 250      ┆ 20266      ┆ Salsa Barbacoa    │
│ FOODS ┆ ADEREZOS ┆ Barbacoa    ┆ NATURA   ┆ 400      ┆ 20325      ┆ Salsa Barbacoa    │
│ FOODS ┆ ADEREZOS ┆ Barbacoa    ┆ NATURA   ┆ 500      ┆ 20503      ┆ Salsa Barbacoa    │
│ FOODS ┆ ADEREZOS ┆ Chimichurri ┆ NATURA   ┆ 350      ┆ 20797      ┆ Chimichurri       │
│ …     ┆ …        ┆ …           ┆ …        ┆ …        ┆ …          ┆ …           

In [5]:
# Agregar cat1, cat2, cat3, brand y sku_size al dataframe original, linkeando las bases por product_id. Luego mostrar el resultado.
df_final = df_final.join(df_productos.select(["product_id", "cat1", "cat2", "cat3", "brand", "sku_size"]), on="product_id", how="left")
print(df_final)

shape: (11_644_324, 13)
┌─────────────┬────────────┬─────────┬──────────────┬───┬─────────────┬─────────┬───────┬──────────┐
│ customer_id ┆ product_id ┆ periodo ┆ plan_precios ┆ … ┆ cat2        ┆ cat3    ┆ brand ┆ sku_size │
│ ---         ┆ ---        ┆ ---     ┆ _cuidados    ┆   ┆ ---         ┆ ---     ┆ ---   ┆ ---      │
│ i64         ┆ i64        ┆ i64     ┆ ---          ┆   ┆ str         ┆ str     ┆ str   ┆ i64      │
│             ┆            ┆         ┆ i64          ┆   ┆             ┆         ┆       ┆          │
╞═════════════╪════════════╪═════════╪══════════════╪═══╪═════════════╪═════════╪═══════╪══════════╡
│ 10001       ┆ 20001      ┆ 201701  ┆ 0            ┆ … ┆ ROPA LAVADO ┆ Liquido ┆ ARIEL ┆ 3000     │
│ 10001       ┆ 20001      ┆ 201702  ┆ 0            ┆ … ┆ ROPA LAVADO ┆ Liquido ┆ ARIEL ┆ 3000     │
│ 10001       ┆ 20001      ┆ 201703  ┆ 0            ┆ … ┆ ROPA LAVADO ┆ Liquido ┆ ARIEL ┆ 3000     │
│ 10001       ┆ 20001      ┆ 201704  ┆ 0            ┆ … ┆ ROPA LAVA

In [6]:
# Agregar totales por categoría (cat1, cat2, cat3) y marca (brand) para cada período.

def agregar_toneladas_jerarquicas(df: pl.DataFrame, /) -> pl.DataFrame:
    """
    Calcula la suma total de toneladas (tn) para las jerarquías individuales,
    las intersecciones de marca por categoría, y el total por producto/período.
    Alineado a la lógica shift(0) para horizontes t+2.
    """
    # 1. Totales generales por jerarquía individual
    columnas_individuales = ["cat1", "cat2", "cat3", "brand"]
    expresiones_individuales = [
        pl.col("tn").sum().over([col, "periodo"]).alias(f"tn_total_{col}")
        for col in columnas_individuales
    ]
    
    # 2. Totales cruzados de la marca dentro de cada categoría
    expresiones_cruces_brand = [
        pl.col("tn").sum().over(["brand", "cat1", "periodo"]).alias("tn_total_brand_en_cat1"),
        pl.col("tn").sum().over(["brand", "cat2", "periodo"]).alias("tn_total_brand_en_cat2"),
        pl.col("tn").sum().over(["brand", "cat3", "periodo"]).alias("tn_total_brand_en_cat3"),
    ]
    
    # 3. NUEVO: Total del producto en el período (agregando todos sus clientes)
    col_tn_producto_mes = [
        pl.col("tn").sum().over(["product_id", "periodo"]).alias("tn_total_producto_mes")
    ]
    
    # Consolidamos todas las expresiones para ejecutarlas en un único bloque optimizado
    todas_las_agregaciones = expresiones_individuales + expresiones_cruces_brand + col_tn_producto_mes
    
    return df.with_columns(todas_las_agregaciones)

In [7]:
df_final = agregar_toneladas_jerarquicas(df_final)

print(df_final)

shape: (11_644_324, 21)
┌────────────┬───────────┬─────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ customer_i ┆ product_i ┆ periodo ┆ plan_prec ┆ … ┆ tn_total_ ┆ tn_total_ ┆ tn_total_ ┆ tn_total_ │
│ d          ┆ d         ┆ ---     ┆ ios_cuida ┆   ┆ brand_en_ ┆ brand_en_ ┆ brand_en_ ┆ producto_ │
│ ---        ┆ ---       ┆ i64     ┆ dos       ┆   ┆ cat1      ┆ cat2      ┆ cat3      ┆ mes       │
│ i64        ┆ i64       ┆         ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│            ┆           ┆         ┆ i64       ┆   ┆ f64       ┆ f64       ┆ f64       ┆ f64       │
╞════════════╪═══════════╪═════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 10001      ┆ 20001     ┆ 201701  ┆ 0         ┆ … ┆ 2655.7867 ┆ 2655.7867 ┆ 1462.9126 ┆ 934.77222 │
│            ┆           ┆         ┆           ┆   ┆ 9         ┆ 9         ┆ 7         ┆           │
│ 10001      ┆ 20001     ┆ 201702  ┆ 0         ┆ … ┆ 2213.7198 ┆ 22

In [8]:
# Se agrega la columna "Agrupation_id" que es la concatenación de customer id y product_id
df_final = df_final.with_columns(
    (pl.col("customer_id") * 100000 + pl.col("product_id")).alias("Agrupation_id")
)

print(df_final)

shape: (11_644_324, 22)
┌────────────┬───────────┬─────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ customer_i ┆ product_i ┆ periodo ┆ plan_prec ┆ … ┆ tn_total_ ┆ tn_total_ ┆ tn_total_ ┆ Agrupatio │
│ d          ┆ d         ┆ ---     ┆ ios_cuida ┆   ┆ brand_en_ ┆ brand_en_ ┆ producto_ ┆ n_id      │
│ ---        ┆ ---       ┆ i64     ┆ dos       ┆   ┆ cat2      ┆ cat3      ┆ mes       ┆ ---       │
│ i64        ┆ i64       ┆         ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ i64       │
│            ┆           ┆         ┆ i64       ┆   ┆ f64       ┆ f64       ┆ f64       ┆           │
╞════════════╪═══════════╪═════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 10001      ┆ 20001     ┆ 201701  ┆ 0         ┆ … ┆ 2655.7867 ┆ 1462.9126 ┆ 934.77222 ┆ 100012000 │
│            ┆           ┆         ┆           ┆   ┆ 9         ┆ 7         ┆           ┆ 1         │
│ 10001      ┆ 20001     ┆ 201702  ┆ 0         ┆ … ┆ 2213.7198 ┆ 13

## Escalado de la variable

Se debe definir si la métrica para escalar es el máximo (max) o la media (mean). El escalado es de la forma x/métrica

Además a la función se le puede pasar muchas columnas para escalar (Todas las agregaciones de toneladas). Supongo que todas arrancan con tn para después usarlo

In [9]:
# Nos quedamos con los primeros 10 productos y los primeros 10 clientes para testear el pipeline
df_final = df_final.filter(
    (pl.col("product_id") <= 20010) & (pl.col("customer_id") <= 10010)
)

In [12]:
metrica = "max"

In [ ]:
# Se crea una función que para cada agupation_id se calcula el rolling max o rolling mean de una columna o varias y luego crea las nuevas columnas con los datos escalados

def calcular_metricas_historicas_multiples(
    df: pl.DataFrame, 
    cols_objetivo: list[str], 
    tipo_metrica: str = "mean", 
    /
) -> pl.DataFrame:
    """
    Calcula una métrica acumulada de TODA la historia disponible agrupando por 
    'Agrupation_id' para una lista de múltiples columnas objetivo, y genera
    sus respectivas columnas escaladas.
    """
    # Ordenamos el dataframe por 'Agrupation_id' y 'periodo' para asegurar la correcta acumulación
    df = df.sort(["Agrupation_id", "periodo"])
    
    # Definimos la expresión acumulada según el tipo
    if tipo_metrica == "mean":
        def expr_acumulada(c): return pl.col(c).cum_mean()
    elif tipo_metrica == "max":
        def expr_acumulada(c): return pl.col(c).cum_max()
    else:
        raise ValueError("El parámetro 'tipo_metrica' debe ser 'mean' o 'max'.")
        
    # Generamos en paralelo las columnas de las métricas hístoricas (.over)
    expresiones_metricas = [
        expr_acumulada(c)
        .over(["Agrupation_id"])
        .cast(pl.Float32)
        .alias(f"{tipo_metrica}_{c}")
        for c in cols_objetivo
    ]
    df = df.with_columns(expresiones_metricas)
    
    # Generamos en paralelo las columnas escaladas (Columna original / Métrica)
    expresiones_escaladas = [
        (pl.col(c) / pl.col(f"{tipo_metrica}_{c}"))
        .fill_nan(0.0)
        .fill_null(0.0)
        .cast(pl.Float32)
        .alias(f"{c}_escalada")
        for c in cols_objetivo
    ]
    df = df.with_columns(expresiones_escaladas)
    
    return df

: 

In [10]:
GRUPO = ["Agrupation_id"]

# Polars NO tiene cum_mean: la media acumulada se calcula como cum_sum / cum_count.
def _cum_mean(expr: pl.Expr) -> pl.Expr:
    return (expr.cum_sum() / expr.cum_count()).over(GRUPO)

# --- Catálogo de escalados ---------------------------------------------------
# Cada escalado recibe el nombre de UNA columna y devuelve una tupla:
#   (metricas, escalada)
#     - metricas: dict {nombre_columna: expr} con las métricas históricas que usa.
#     - escalada: expr con la columna escalada final.
# Para agregar uno nuevo: copiá una función de acá, cambiá la fórmula, y sumala
# al diccionario ESCALADORES de abajo.

def sobre_max(c):
    m = pl.col(c).cum_max().over(GRUPO)
    return {f"max_{c}": m}, pl.col(c) / m

def sobre_min(c):
    m = pl.col(c).cum_min().over(GRUPO)
    return {f"min_{c}": m}, pl.col(c) / m

def sobre_media(c):
    m = _cum_mean(pl.col(c))
    return {f"media_{c}": m}, pl.col(c) / m

def desvio_relativo(c):                      # (x - media) / media
    media = _cum_mean(pl.col(c))
    return {f"media_{c}": media}, (pl.col(c) - media) / media

def zscore(c):                               # (x - media) / desvío
    x = pl.col(c)
    media = _cum_mean(x)
    std = (_cum_mean(x ** 2) - media ** 2).sqrt()
    return {f"media_{c}": media, f"std_{c}": std}, (x - media) / std

# El "menú" que ve el usuario final:
ESCALADORES = {
    "sobre_max":        sobre_max,
    "sobre_min":        sobre_min,
    "sobre_media":      sobre_media,
    "desvio_relativo":  desvio_relativo,
    "zscore":           zscore,
}

# --- Función principal -------------------------------------------------------

def calcular_metricas_historicas_multiples(
    df: pl.DataFrame,
    cols_objetivo: list[str],
    metrica: str = "sobre_media",
    sufijo: str | None = None,
    incluir_metrica: bool = False,
    /,
) -> pl.DataFrame:
    """
    Genera columnas escaladas por 'Agrupation_id' usando la acumulación de toda
    la historia. 'metrica' elige el tipo de escalado del catálogo ESCALADORES.
    Si 'incluir_metrica' es True, además agrega las columnas de métrica histórica
    (ej. max_<col>, media_<col>) como features independientes.
    """
    if metrica not in ESCALADORES:
        raise ValueError(f"'metrica' debe ser uno de: {list(ESCALADORES)}")

    escalador = ESCALADORES[metrica]
    sufijo = sufijo or metrica               # por defecto la columna se llama <col>_<metrica>

    df = df.sort(["Agrupation_id", "periodo"])
    expresiones = []
    for c in cols_objetivo:
        metricas, escalada = escalador(c)

        if incluir_metrica:
            expresiones += [
                expr.cast(pl.Float32).alias(nombre)
                for nombre, expr in metricas.items()
            ]

        expresiones.append(
            escalada
            .fill_nan(0.0)
            .fill_null(0.0)
            .cast(pl.Float32)
            .alias(f"{c}_{sufijo}")
        )

    return df.with_columns(expresiones)

In [15]:
df_final =calcular_metricas_historicas_multiples(df_final, ["tn", "tn_total_producto_mes"], "sobre_media", None, True)

In [13]:
print(df_final.head(36))

shape: (36, 26)
┌────────────┬───────────┬─────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ customer_i ┆ product_i ┆ periodo ┆ plan_prec ┆ … ┆ max_tn    ┆ tn_sobre_ ┆ max_tn_to ┆ tn_total_ │
│ d          ┆ d         ┆ ---     ┆ ios_cuida ┆   ┆ ---       ┆ max       ┆ tal_produ ┆ producto_ │
│ ---        ┆ ---       ┆ i64     ┆ dos       ┆   ┆ f32       ┆ ---       ┆ cto_mes   ┆ mes_sobre │
│ i64        ┆ i64       ┆         ┆ ---       ┆   ┆           ┆ f32       ┆ ---       ┆ _ma…      │
│            ┆           ┆         ┆ i64       ┆   ┆           ┆           ┆ f32       ┆ ---       │
│            ┆           ┆         ┆           ┆   ┆           ┆           ┆           ┆ f32       │
╞════════════╪═══════════╪═════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 10001      ┆ 20001     ┆ 201701  ┆ 0         ┆ … ┆ 99.438606 ┆ 1.0       ┆ 934.77221 ┆ 1.0       │
│            ┆           ┆         ┆           ┆   ┆           ┆           

In [12]:
df_final =calcular_metricas_historicas_multiples(df_final, ["tn", "tn_total_producto_mes"], metrica)

print(df_final[["Agrupation_id", "periodo", "tn", "max_tn", "tn_escalada", "tn_total_producto_mes", "max_tn_total_producto_mes", "tn_total_producto_mes_escalada"]].head(36))

shape: (36, 8)
┌────────────┬─────────┬───────────┬────────────┬────────────┬────────────┬────────────┬───────────┐
│ Agrupation ┆ periodo ┆ tn        ┆ max_tn     ┆ tn_escalad ┆ tn_total_p ┆ max_tn_tot ┆ tn_total_ │
│ _id        ┆ ---     ┆ ---       ┆ ---        ┆ a          ┆ roducto_me ┆ al_product ┆ producto_ │
│ ---        ┆ i64     ┆ f64       ┆ f32        ┆ ---        ┆ s          ┆ o_mes      ┆ mes_escal │
│ i64        ┆         ┆           ┆            ┆ f32        ┆ ---        ┆ ---        ┆ ada       │
│            ┆         ┆           ┆            ┆            ┆ f64        ┆ f32        ┆ ---       │
│            ┆         ┆           ┆            ┆            ┆            ┆            ┆ f32       │
╞════════════╪═════════╪═══════════╪════════════╪════════════╪════════════╪════════════╪═══════════╡
│ 1000120001 ┆ 201701  ┆ 99.43861  ┆ 99.438606  ┆ 1.0        ┆ 934.77222  ┆ 934.772217 ┆ 1.0       │
│ 1000120001 ┆ 201702  ┆ 198.84365 ┆ 198.843643 ┆ 1.0        ┆ 798.0162   ┆ 

## Variable Clase (Revisar que hacer cuando el max o mean sean 0, nacimiento de productos)

Se genera la variable a predecir que pueden ser 2 opciones, el valor del mes +2 o el delta entre el mes +2 y el valor actual. En ambos casos esta nueva variable se la escala dividiendola por la métrica calculada en el mes actual

In [13]:
tipo_clase = "delta"

In [14]:
# Se crea la función para crear el campo clase y escalarlo con el mismo criterio

def crear_variable_clase_escalada(
    df: pl.DataFrame, 
    col_objetivo: str, 
    tipo_clase: str, 
    tipo_metrica: str = "mean", 
    /
) -> pl.DataFrame:
    """
    Genera y escala la variable a predecir (target) para el horizonte t+2 
    según el tipo de clase solicitado (value o delta).
    
    Deja los valores faltantes como null para que LightGBM/XGBoost los maneje de forma nativa.
    """
    # Ordenamos el dataset
    df = df.sort(["Agrupation_id", "periodo"])
    
    # Validamos que la columna de la métrica exista
    col_metrica_actual = f"{tipo_metrica}_{col_objetivo}"
    if col_metrica_actual not in df.columns:
        raise KeyError(
            f"La columna métrica '{col_metrica_actual}' no se encuentra en el DataFrame. "
            f"Asegurate de correr primero 'calcular_metricas_historicas_multiples'."
        )
    
    # Obtenemos el valor del mes +2 usando shift negativo (-2) sobre el grupo
    col_futura = pl.col(col_objetivo).shift(-2).over(["Agrupation_id"])
    
    # Evaluamos el parámetro 'tipo_clase' y construimos la expresión correspondiente
    # Protegemos contra la división por cero mapeando a null en ese escenario
    if tipo_clase == "value":
        expr_target = (
            pl.when(pl.col(col_metrica_actual) > 0)
            .then(col_futura / pl.col(col_metrica_actual))
            .otherwise(None)
        )
    
        
    elif tipo_clase == "delta":
        expr_target = (
            pl.when(pl.col(col_metrica_actual) > 0)
            .then((col_futura - pl.col(col_objetivo)) / pl.col(col_metrica_actual))
            .otherwise(None)
        )
        
    else:
        raise ValueError("El parámetro 'tipo_clase' debe ser exclusivamente 'value' o 'delta'.")
    
    # Sumamos la columna elegida en formato Float32
    return df.with_columns(
        expr_target.cast(pl.Float32).alias(f"Clase_{tipo_clase}")
    )

In [15]:
df_final = crear_variable_clase_escalada(df_final, "tn", tipo_clase, metrica)

print(df_final[["Agrupation_id", "periodo", "tn", "max_tn", "tn_escalada", f"Clase_{tipo_clase}"]].head(36))

shape: (36, 6)
┌───────────────┬─────────┬───────────┬────────────┬─────────────┬─────────────┐
│ Agrupation_id ┆ periodo ┆ tn        ┆ max_tn     ┆ tn_escalada ┆ Clase_delta │
│ ---           ┆ ---     ┆ ---       ┆ ---        ┆ ---         ┆ ---         │
│ i64           ┆ i64     ┆ f64       ┆ f32        ┆ f32         ┆ f32         │
╞═══════════════╪═════════╪═══════════╪════════════╪═════════════╪═════════════╡
│ 1000120001    ┆ 201701  ┆ 99.43861  ┆ 99.438606  ┆ 1.0         ┆ -0.070126   │
│ 1000120001    ┆ 201702  ┆ 198.84365 ┆ 198.843643 ┆ 1.0         ┆ -0.933127   │
│ 1000120001    ┆ 201703  ┆ 92.46537  ┆ 198.843643 ┆ 0.465015    ┆ 0.04295     │
│ 1000120001    ┆ 201704  ┆ 13.29728  ┆ 198.843643 ┆ 0.066873    ┆ 0.57709     │
│ 1000120001    ┆ 201705  ┆ 101.00563 ┆ 198.843643 ┆ 0.507965    ┆ 0.001013    │
│ …             ┆ …       ┆ …         ┆ …          ┆ …           ┆ …           │
│ 1000120001    ┆ 201908  ┆ 33.63991  ┆ 439.906464 ┆ 0.076471    ┆ 0.323682    │
│ 1000120001 

## Feature engineering histórico

Este se puede usar sobre todas las variables de tn sea cual sea su nivel de agregación y estén o no escaladas

In [16]:

def crear_features_temporales_multiples(df: pl.DataFrame, cols_objetivo: list[str], /) -> pl.DataFrame:
    """
    Genera Lags, Deltas, Estadísticas Móviles (2-6) y rachas de tendencia (crece/cae)
    en paralelo para una lista de múltiples variables, agrupando siempre por Agrupation_id.
    """
    # 1. Aseguramos el orden cronológico por la clave única y período
    grupo = ["Agrupation_id"]
    df = df.sort(grupo + ["periodo"])
    
    # --- FASE 1: CONSTRUCCIÓN DE EXPRESIONES EN PARALELO ---
    lags = []
    deltas = []
    medias_moviles = []
    desvios_moviles = []
    maximos_moviles = []
    minimos_moviles = []
    auxiliares_rachas = []
    
    for col in cols_objetivo:
        # Lags y Deltas (1 a 12)
        lags.extend([pl.col(col).shift(i).over(grupo).alias(f"{col}_lag_{i}") for i in range(1, 13)])
        deltas.extend([(pl.col(col).shift(i) - pl.col(col).shift(i + 1)).over(grupo).alias(f"{col}_delta_lag_{i}") for i in range(1, 13)])
        
        # Estadísticas Móviles (2 a 6)
        medias_moviles.extend([pl.col(col).rolling_mean(window_size=i).over(grupo).alias(f"{col}_roll_mean_{i}") for i in range(2, 7)])
        desvios_moviles.extend([pl.col(col).rolling_std(window_size=i).over(grupo).alias(f"{col}_roll_std_{i}") for i in range(2, 7)])
        maximos_moviles.extend([pl.col(col).rolling_max(window_size=i).over(grupo).alias(f"{col}_roll_max_{i}") for i in range(2, 7)])
        minimos_moviles.extend([pl.col(col).rolling_min(window_size=i).over(grupo).alias(f"{col}_roll_min_{i}") for i in range(2, 7)])
        
        # Identificadores de bloques para rachas macro
        viene_creciendo = (pl.col(col) > pl.col(col).shift(1)).cast(pl.Int8)
        auxiliares_rachas.append((viene_creciendo != viene_creciendo.shift(1)).fill_null(True).cum_sum().over(grupo).alias(f"_id_crece_{col}"))
        
        viene_cayendo = (pl.col(col) < pl.col(col).shift(1)).cast(pl.Int8)
        auxiliares_rachas.append((viene_cayendo != viene_cayendo.shift(1)).fill_null(True).cum_sum().over(grupo).alias(f"_id_cae_{col}"))

    # Consolidamos y ejecutamos la primera gran tanda de features
    expresiones_fase1 = lags + deltas + medias_moviles + desvios_moviles + maximos_moviles + minimos_moviles + auxiliares_rachas
    df = df.with_columns(expresiones_fase1)
    
    # --- FASE 2: CÁLCULO DE RACHAS CON BLOQUES YA MATERIALIZADOS ---
    expresiones_rachas = []
    columnas_a_borrar = []
    
    for col in cols_objetivo:
        id_crece = f"_id_crece_{col}"
        id_cae = f"_id_cae_{col}"
        columnas_a_borrar.extend([id_crece, id_cae])
        
        # Racha de crecimiento
        expr_crece = (
            pl.when(pl.col(col) > pl.col(col).shift(1))
            .then(pl.int_range(0, pl.len()).over(grupo + [id_crece]) + 1)
            .otherwise(0)
            .alias(f"{col}_racha_meses_creciendo")
        )
        # Racha de caída
        expr_cae = (
            pl.when(pl.col(col) < pl.col(col).shift(1))
            .then(pl.int_range(0, pl.len()).over(grupo + [id_cae]) + 1)
            .otherwise(0)
            .alias(f"{col}_racha_meses_cayendo")
        )
        expresiones_rachas.extend([expr_crece, expr_cae])
        
    df = df.with_columns(expresiones_rachas)
    
    # Eliminamos las columnas auxiliares indexadas de control
    return df.drop(columnas_a_borrar)

In [17]:
variables = ["tn", "tn_total_producto_mes"]

df_final = crear_features_temporales_multiples(df_final, variables)

print(df_final)

shape: (3_456, 119)
┌────────────┬───────────┬─────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ customer_i ┆ product_i ┆ periodo ┆ plan_prec ┆ … ┆ tn_racha_ ┆ tn_racha_ ┆ tn_total_ ┆ tn_total_ │
│ d          ┆ d         ┆ ---     ┆ ios_cuida ┆   ┆ meses_cre ┆ meses_cay ┆ producto_ ┆ producto_ │
│ ---        ┆ ---       ┆ i64     ┆ dos       ┆   ┆ ciendo    ┆ endo      ┆ mes_racha ┆ mes_racha │
│ i64        ┆ i64       ┆         ┆ ---       ┆   ┆ ---       ┆ ---       ┆ _me…      ┆ _me…      │
│            ┆           ┆         ┆ i64       ┆   ┆ i64       ┆ i64       ┆ ---       ┆ ---       │
│            ┆           ┆         ┆           ┆   ┆           ┆           ┆ i64       ┆ i64       │
╞════════════╪═══════════╪═════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 10001      ┆ 20001     ┆ 201701  ┆ 0         ┆ … ┆ 0         ┆ 0         ┆ 0         ┆ 0         │
│ 10001      ┆ 20001     ┆ 201702  ┆ 0         ┆ … ┆ 1         ┆ 0     

## Feature Engineering Variable original

Este bloque está pensado para hacer feature engineering sobre la variable original y contruyendo variables de tipo rachas desde que se está comprando, racha desde que es 0 el valor, primes mes de compra, si compro lo mismo que el mes anterior, racha meses que compra lo mismo, meses que pasaron desde la primer compra (primer período en que tn no es cero)

In [20]:

def crear_features_comportamiento_multiples(df: pl.DataFrame, cols_objetivo: list[str], /) -> pl.DataFrame:
    """
    Genera variables de comportamiento y rachas históricas individuales (meses activo, 
    rachas de ceros, rachas de estabilidad y coincidencia mensual) para una lista 
    de múltiples columnas, agrupando siempre por Agrupation_id.
    Alineado a la lógica shift(0) para horizontes t+2.
    """
    # 1. Aseguramos el orden cronológico estricto por la llave única
    grupo = ["Agrupation_id"]
    df = df.sort(grupo + ["periodo"])
    
    # Precalculamos el vector de tiempo lineal una sola vez para ahorrar CPU
    meses_lineales_actual = (pl.col("periodo") // 100) * 12 + (pl.col("periodo") % 100)
    
    expresiones_fase1 = []
    auxiliares_rachas = []
    
    # --- FASE 1: CONSTRUCCIÓN DE EXPRESIONES EN PARALELO ---
    for col in cols_objetivo:
        # A. ¿Compró exactamente lo mismo que el mes anterior?
        compro_mismo_mes_anterior = (
            pl.when(pl.col(col) == pl.col(col).shift(1))
            .then(1)
            .otherwise(0)
            .over(grupo)
            .cast(pl.Int8)
            .alias(f"{col}_compro_mismo_mes_anterior")
        )
        expresiones_fase1.append(compro_mismo_mes_anterior)
        
        # B. Meses que pasaron desde la primera compra real (primer período > 0)
        periodo_primer_venta_real = pl.when(pl.col(col) > 0).then(pl.col("periodo")).otherwise(None)
        min_periodo_real = periodo_primer_venta_real.min().over(grupo)
        meses_lineales_primer_venta = (min_periodo_real // 100) * 12 + (min_periodo_real % 100)
        
        meses_desde_primer_venta = (
            pl.when(meses_lineales_actual >= meses_lineales_primer_venta)
            .then(meses_lineales_actual - meses_lineales_primer_venta)
            .otherwise(0)
            .over(grupo)
            .cast(pl.Int32)
            .alias(f"{col}_meses_desde_primer_compra")
        )
        expresiones_fase1.append(meses_desde_primer_venta)
        
        # C. Identificadores de bloques para construir las rachas exactas
        _cambio_valor = (pl.col(col) != pl.col(col).shift(1)).fill_null(True).cum_sum().over(grupo).alias(f"_id_bloque_valor_{col}")
        
        es_cero = (pl.col(col) == 0).cast(pl.Int8)
        _rompe_racha_cero = (es_cero != es_cero.shift(1)).fill_null(True).cum_sum().over(grupo).alias(f"_id_bloque_cero_{col}")
        
        es_activo = (pl.col(col) > 0).cast(pl.Int8)
        _rompe_racha_activa = (es_activo != es_activo.shift(1)).fill_null(True).cum_sum().over(grupo).alias(f"_id_bloque_activo_{col}")
        
        auxiliares_rachas.extend([_cambio_valor, _rompe_racha_cero, _rompe_racha_activa])
        
    # Ejecutamos la primera tanda de transformaciones
    df = df.with_columns(expresiones_fase1 + auxiliares_rachas)
    
    # --- FASE 2: CÁLCULO DE CONTADORES DE RACHAS CON BLOQUES MATERIALIZADOS ---
    expresiones_fase2 = []
    columnas_a_borrar = []
    
    for col in cols_objetivo:
        id_valor = f"_id_bloque_valor_{col}"
        id_cero = f"_id_bloque_cero_{col}"
        id_activo = f"_id_bloque_activo_{col}"
        columnas_a_borrar.extend([id_valor, id_cero, id_activo])
        
        # Racha meses que compra exactamente lo mismo
        racha_mismo_valor = (
            pl.int_range(0, pl.len()).over(grupo + [id_valor]).cast(pl.Int32)
            .alias(f"{col}_racha_meses_mismo_valor")
        )
        
        # Racha desde que el valor es cero (meses consecutivos sin comprar)
        racha_cero = (
            pl.when(pl.col(col) == 0)
            .then(pl.int_range(0, pl.len()).over(grupo + [id_cero]) + 1)
            .otherwise(0)
            .cast(pl.Int32)
            .alias(f"{col}_racha_meses_cero")
        )
        
        # Racha meses consecutivos comprando (meses activo consecutivos)
        racha_activa = (
            pl.when(pl.col(col) > 0)
            .then(pl.int_range(0, pl.len()).over(grupo + [id_activo]) + 1)
            .otherwise(0)
            .cast(pl.Int32)
            .alias(f"{col}_racha_meses_activo_consecutivo")
        )
        
        expresiones_fase2.extend([racha_mismo_valor, racha_cero, racha_activa])
        
    df = df.with_columns(expresiones_fase2)
    
    # Eliminamos las columnas auxiliares indexadas de control para limpiar la RAM
    return df.drop(columnas_a_borrar)

In [21]:
df_final = crear_features_comportamiento_multiples(df_final, ["tn"])

print(df_final.head(36))

shape: (36, 124)
┌────────────┬───────────┬─────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ customer_i ┆ product_i ┆ periodo ┆ plan_prec ┆ … ┆ tn_meses_ ┆ tn_racha_ ┆ tn_racha_ ┆ tn_racha_ │
│ d          ┆ d         ┆ ---     ┆ ios_cuida ┆   ┆ desde_pri ┆ meses_mis ┆ meses_cer ┆ meses_act │
│ ---        ┆ ---       ┆ i64     ┆ dos       ┆   ┆ mer_compr ┆ mo_valor  ┆ o         ┆ ivo_conse │
│ i64        ┆ i64       ┆         ┆ ---       ┆   ┆ a         ┆ ---       ┆ ---       ┆ cut…      │
│            ┆           ┆         ┆ i64       ┆   ┆ ---       ┆ i32       ┆ i32       ┆ ---       │
│            ┆           ┆         ┆           ┆   ┆ i32       ┆           ┆           ┆ i32       │
╞════════════╪═══════════╪═════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 10001      ┆ 20001     ┆ 201701  ┆ 0         ┆ … ┆ 0         ┆ 0         ┆ 0         ┆ 1         │
│ 10001      ┆ 20001     ┆ 201702  ┆ 0         ┆ … ┆ 1         ┆ 0        

## Feature Engineering de Marketshare

La idea acá es poder definir una columna que refiera a producto y analizar el market share de ese producto con una categoría padre. También sería extrapolable para analizar el market share y dominancia de un producto a nivel cliente pero no lo hice en la parte de preprocesamiento

In [27]:
import polars as pl

def calcular_market_share_y_dominancia_completo(
    df: pl.DataFrame, 
    config_cruces: list[tuple[str, str, list[str]]], 
    /
) -> pl.DataFrame:
    """
    Calcula dinámicamente el conteo de SKUs activos del segmento, las toneladas totales 
    del segmento, el Market Share Real y la dummy de dominancia para múltiples cruces en paralelo.
    Todo agrupado de forma consistente y devuelto a nivel de Agrupation_id.
    """
    # 1. Aseguramos el orden secuencial estricto por la clave de la relación cliente-producto
    df = df.sort(["Agrupation_id", "periodo"])
    
    expresiones_segmentos = []
    expresiones_ms = []
    expresiones_dummies = []
    
    # --- PASO 1: Calcular Toneladas Totales y Conteo de SKUs Activos para cada segmento ---
    for col_producto, col_tn_segmento, cols_grupo_segmento in config_cruces:
        # La llave de la ventana temporal siempre cruza las columnas del segmento con el 'periodo' actual
        llave_ventana = cols_grupo_segmento + ["periodo"]
        sufijo = f"{col_producto}_vs_{col_tn_segmento}"
        
        # A. Toneladas totales del segmento mensual
        expr_tn_seg = pl.col(col_producto).sum().over(llave_ventana).alias(col_tn_segmento)
        
        # B. Conteo dinámico de productos activos (tn > 0) en este segmento mensual
        # NOTA: Usamos la columna base 'tn' para el filtro de actividad del mercado real
        expr_prods_seg = (
            pl.col("product_id")
            .filter(pl.col("tn") > 0)
            .n_unique()
            .over(llave_ventana)
            .alias(f"prods_activos_{sufijo}")
        )
        
        expresiones_segmentos.extend([expr_tn_seg, expr_prods_seg])
        
    df = df.with_columns(expresiones_segmentos)
    
    # --- PASO 2: Calcular Market Share Real y Dummies de Dominancia ---
    for col_producto, col_tn_segmento, _ in config_cruces:
        sufijo = f"{col_producto}_vs_{col_tn_segmento}"
        col_prods_activos = f"prods_activos_{sufijo}"
        nombre_ms_real = f"ms_real_{sufijo}"
        
        # A. Market Share Real = Columna Original / Columna del Segmento
        # Deja null si el volumen del segmento es 0 o null
        col_ms_real = (
            pl.when(pl.col(col_tn_segmento) > 0)
            .then(pl.col(col_producto) / pl.col(col_tn_segmento))
            .otherwise(None)
        ).alias(nombre_ms_real)
        expresiones_ms.append(col_ms_real)
        
        # B. Market Share Teórico Equitativo = 1 / Cantidad de Productos Activos
        col_ms_teorico = (
            pl.when(pl.col(col_prods_activos) > 0)
            .then(1.0 / pl.col(col_prods_activos))
            .otherwise(None)
        )
        
        # C. CORREGIDO: Dummy de Dominancia = ¿El MS Real supera la cuota equitativa?
        # Llamamos directamente a la columna que se creará en la fase anterior mediante su String.
        # Polars propaga los nulls nativamente si el MS o el teórico son nulos.
        dummy_dominancia = (
            pl.when(pl.col(nombre_ms_real) > col_ms_teorico)
            .then(1)
            .otherwise(0)
            .cast(pl.Int8)
            .alias(f"dummy_lider_{sufijo}")
        )
        expresiones_dummies.append(dummy_dominancia)
        
    # Consolidamos los Market Shares en paralelo casteando a Float32 para optimizar la RAM
    df = df.with_columns(expresiones_ms)
    columnas_ms_creadas = [f"ms_real_{c[0]}_vs_{c[1]}" for c in config_cruces]
    df = df.with_columns([pl.col(c).cast(pl.Float32) for c in columnas_ms_creadas])
    
    # Acoplamos las dummies finales de dominancia
    return df.with_columns(expresiones_dummies)

In [28]:
# Configuramos los cruces que queremos que la función arme desde cero
mis_cruces_config = [
    # Caso 1: Comparar tn del producto contra su categoría macro (cat1)
    ("tn_total_producto_mes", "tn_total_cat1", ["cat1"]),
    
    # Caso 2: Comparar tn del producto contra su subcategoría específica (cat3)
    ("tn_total_producto_mes", "tn_total_cat3", ["cat3"]),
    
    # Caso 3: Comparar tn del producto contra la intersección de su marca en esa cat2
    ("tn_total_producto_mes", "tn_total_brand_en_cat2", ["brand", "cat2"])
]

# Ejecutamos la función sobre el dataset
df_final = calcular_market_share_y_dominancia_completo(df_final, mis_cruces_config)

print(df_final.head())

shape: (5, 133)
┌────────────┬───────────┬─────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ customer_i ┆ product_i ┆ periodo ┆ plan_prec ┆ … ┆ ms_real_t ┆ dummy_lid ┆ dummy_lid ┆ dummy_lid │
│ d          ┆ d         ┆ ---     ┆ ios_cuida ┆   ┆ n_total_p ┆ er_tn_tot ┆ er_tn_tot ┆ er_tn_tot │
│ ---        ┆ ---       ┆ i64     ┆ dos       ┆   ┆ roducto_m ┆ al_produc ┆ al_produc ┆ al_produc │
│ i64        ┆ i64       ┆         ┆ ---       ┆   ┆ es_…      ┆ to_…      ┆ to_…      ┆ to_…      │
│            ┆           ┆         ┆ i64       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│            ┆           ┆         ┆           ┆   ┆ f32       ┆ i8        ┆ i8        ┆ i8        │
╞════════════╪═══════════╪═════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 10001      ┆ 20001     ┆ 201701  ┆ 0         ┆ … ┆ 0.1       ┆ 0         ┆ 0         ┆ 0         │
│ 10001      ┆ 20001     ┆ 201702  ┆ 0         ┆ … ┆ 0.1       ┆ 0         